# 03. Positional Encoding, Schedule, Complexity와 검증 Gate

## 학습 목표

- sinusoidal positional encoding을 vectorized하게 구현한다.
- 논문의 warmup learning-rate schedule peak를 확인한다.
- dense attention score memory가 sequence length의 제곱으로 증가함을 계산한다.
- 논문 식을 제품 code로 옮길 때 필요한 자동 검증 gate를 정의한다.

In [ ]:
# numpy는 삼각함수, vectorization과 수치 검증을 제공한다.
import numpy as np

# 재현 가능한 scaling 실험용 random generator를 만든다.
rng = np.random.default_rng(seed=3000)
# 작은 matrix를 읽기 쉽게 출력한다.
np.set_printoptions(precision=4, suppress=True)

## Sinusoidal positional encoding

`angles[pos, i] = pos / 10000^(2 floor(i/2) / d_model)`을 먼저 만들고 짝수 차원에는 sine, 홀수 차원에는 cosine을 적용한다.

In [ ]:
# sequence length와 model dimension을 받아 위치 행렬을 만든다.
def positional_encoding(length: int, d_model: int) -> np.ndarray:
    # sine/cosine pair를 만들기 위해 d_model은 짝수여야 한다.
    assert d_model > 0 and d_model % 2 == 0
    # position을 [length, 1] column vector로 만든다.
    positions = np.arange(length, dtype=np.float64)[:, None]
    # 짝수 차원 index 0, 2, 4, ...를 row vector로 만든다.
    even_dimensions = np.arange(0, d_model, 2, dtype=np.float64)[None, :]
    # 차원마다 다른 주파수를 만들기 위한 denominator다.
    denominators = np.power(10000.0, even_dimensions / float(d_model))
    # 모든 position과 주파수 조합의 angle을 broadcasting으로 계산한다.
    angles = positions / denominators
    # 결과 행렬을 0으로 초기화한다.
    encoding = np.zeros((length, d_model), dtype=np.float64)
    # 짝수 차원에는 sine을 기록한다.
    encoding[:, 0::2] = np.sin(angles)
    # 홀수 차원에는 같은 angle의 cosine을 기록한다.
    encoding[:, 1::2] = np.cos(angles)
    # embedding에 더할 [length, d_model] matrix를 반환한다.
    return encoding

# 길이 6, 차원 8의 작은 위치 encoding을 만든다.
pe = positional_encoding(length=6, d_model=8)
# position별 pattern을 관찰한다.
print(pe)
# shape는 token 위치 수 × model dimension이어야 한다.
assert pe.shape == (6, 8)
# position 0의 sine 차원은 모두 sin(0)=0이다.
assert np.allclose(pe[0, 0::2], 0.0)
# position 0의 cosine 차원은 모두 cos(0)=1이다.
assert np.allclose(pe[0, 1::2], 1.0)
# 모든 sine/cosine 값은 -1과 1 사이여야 한다.
assert np.max(np.abs(pe)) <= 1.0

## 논문 learning-rate schedule

0 step은 음의 지수 때문에 정의되지 않으므로 training step은 1부터 시작한다. warmup 구간은 증가하고 그 뒤에는 감소해야 한다.

In [ ]:
# 논문 식 (3)의 learning rate multiplier를 계산한다.
def transformer_learning_rate(step: int, d_model: int, warmup_steps: int) -> float:
    # step 0과 잘못된 dimension은 수식 정의 밖이므로 거부한다.
    assert step >= 1 and d_model > 0 and warmup_steps > 0
    # warmup 이후 사용하는 inverse-square-root 항이다.
    decay_term = step ** -0.5
    # warmup 동안 선형 증가하는 항이다.
    warmup_term = step * (warmup_steps ** -1.5)
    # model dimension scaling과 두 schedule 중 작은 값을 곱한다.
    return (d_model ** -0.5) * min(decay_term, warmup_term)

# 논문 base dimension과 warmup 4000을 사용한다.
probe_steps = [1, 1000, 2000, 4000, 8000, 16000]
# 각 관찰 step의 learning rate를 계산한다.
rates = [transformer_learning_rate(step, 512, 4000) for step in probe_steps]
# step과 rate를 함께 출력해 peak 위치를 읽는다.
for step, rate in zip(probe_steps, rates):
    # scientific notation으로 작은 learning rate를 표시한다.
    print(f'step={step:5d} rate={rate:.8e}')
# warmup 끝까지는 관찰 rate가 계속 증가해야 한다.
assert rates[0] < rates[1] < rates[2] < rates[3]
# warmup 뒤 8000과 16000 step에서는 감소해야 한다.
assert rates[3] > rates[4] > rates[5]

## Attention score memory

아래 계산은 Q·K·V, activation, gradient와 allocator overhead를 제외하고 score tensor 하나만 계산한다. 실제 training memory는 더 크다.

In [ ]:
# dense attention score tensor의 byte 수를 계산한다.
def score_memory_bytes(
    batch_size: int,
    num_heads: int,
    sequence_length: int,
    bytes_per_element: int,
) -> int:
    # score shape는 [batch, head, query_length, key_length]다.
    element_count = batch_size * num_heads * sequence_length * sequence_length
    # 원소 수에 dtype byte 크기를 곱한다.
    return element_count * bytes_per_element

# FP16/BF16처럼 원소당 2 byte인 score를 가정한다.
sequence_lengths = [128, 512, 2048, 8192]
# 각 길이에 대해 batch 1, head 8의 score memory를 계산한다.
for length in sequence_lengths:
    # byte를 2^20으로 나눠 MiB 단위로 바꾼다.
    memory_mib = score_memory_bytes(1, 8, length, 2) / (1024.0 ** 2)
    # sequence length와 score tensor 하나의 크기를 출력한다.
    print(f'length={length:5d} score_only={memory_mib:9.2f} MiB')
# 길이를 두 배로 하면 score 원소 수는 네 배가 되어야 한다.
base = score_memory_bytes(1, 8, 512, 2)
# 1024 길이의 결과를 비교한다.
doubled = score_memory_bytes(1, 8, 1024, 2)
# dense attention의 quadratic 관계를 assertion으로 고정한다.
assert doubled == 4 * base

## Scaling ablation

여러 `d_k`에서 raw score 표준편차와 scaling된 score 표준편차를 비교한다. 독립 표준정규 가정 아래 scaling된 값은 대략 1 부근을 유지해야 한다.

In [ ]:
# 여러 key dimension을 순회한다.
for d_k in [8, 64, 512]:
    # query 4096개를 표준정규분포에서 생성한다.
    q = rng.normal(size=(4096, d_k))
    # 각 query와 짝이 되는 key를 독립적으로 생성한다.
    k = rng.normal(size=(4096, d_k))
    # 같은 행의 q와 k를 곱해 더한 dot product를 만든다.
    raw = np.sum(q * k, axis=-1)
    # 논문 방식대로 sqrt(d_k)로 나눈다.
    scaled = raw / np.sqrt(float(d_k))
    # raw와 scaled 표준편차를 비교 출력한다.
    print(f'd_k={d_k:3d} raw_std={raw.std():.3f} scaled_std={scaled.std():.3f}')
    # 충분한 sample에서 scaled 표준편차가 1 근처인지 넓은 tolerance로 검사한다.
    assert 0.85 < scaled.std() < 1.15

## 제품 적용 전 검증 gate

| Gate | 실패 시 의미 |
|---|---|
| Q·K `d_k` 일치 | matrix multiplication contract 오류 |
| K·V sequence length 일치 | weight와 value 위치 불일치 |
| `d_model % num_heads == 0` | head reshape 불가능 |
| fully-masked row 없음 또는 별도 처리 | softmax NaN 가능 |
| causal upper triangle 0 | 미래 정보 누출 |
| weight row sum ≈ 1 | softmax 축·mask 오류 |
| finite Q·K·V·output | overflow, invalid input 또는 kernel 오류 |
| reference와 optimized kernel 오차 이내 | backend 변환 오류 |

실제 배포에서는 shape만 맞는 것으로 끝내지 않고 Python reference, framework kernel, ONNX/TensorRT 같은 target runtime을 같은 golden vector로 비교한다.

In [ ]:
# 최종 self-check에 사용할 작은 probability matrix를 만든다.
example_weights = np.array([[0.2, 0.3, 0.5], [1.0, 0.0, 0.0]])
# 모든 값이 finite인지 확인한다.
assert np.all(np.isfinite(example_weights))
# probability가 음수가 아닌지 확인한다.
assert np.all(example_weights >= 0.0)
# 각 query row의 합이 1인지 확인한다.
assert np.allclose(example_weights.sum(axis=-1), 1.0)
# 모든 심화 실습 gate를 통과했음을 표시한다.
print('advanced attention checks passed')